In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/My_Safe_Folder/FinBuddy_unbanked')

Mounted at /content/drive


In [2]:
import os
# Check if Drive is mounted and list its contents to locate 'matching_engine.py' and 'script_rag.py'
if os.path.exists('/content/drive/MyDrive/My_Safe_Folder/FinBuddy_unbanked'):
    print("--- Files in Google Drive root ('/content/drive/MyDrive'): ---")
    try:
        display(os.listdir('/content/drive/MyDrive/My_Safe_Folder/FinBuddy_unbanked'))
    except Exception as e:
        print(f"Could not list files: {e}")
else:
    print("Google Drive is not mounted at '/content/drive/MyDrive'. Please make sure the first cell completed successfully.")

--- Files in Google Drive root ('/content/drive/MyDrive'): ---


['upi_agents_mvp.py',
 'matching_engine.py',
 'test_agents.py',
 'script_rag.py',
 '__pycache__']

In [3]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [4]:
!pip install -q langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.


In [5]:
import sys
import site
# Force python to refresh its global package directories
importlib_reload = importlib_reload if 'importlib_reload' in globals() else None
try:
    from importlib import reload
    import site
    reload(site)
except Exception:
    pass

In [6]:
import sys
# Force clearing out-of-date import cache for langchain integrations
sys.modules.pop('langchain_google_genai', None)

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    print("SUCCESS: langchain_google_genai is successfully imported and available!")
except ImportError as e:
    print(f"FAIL: Import still failing with: {e}")

SUCCESS: langchain_google_genai is successfully imported and available!


In [7]:
import argparse, json, operator, os
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Annotated, TypedDict

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from matching_engine import CLUSTER_FEATURES, FEATURES, MIN_REPAY_SHARE, POLICY, REASON_TEXT
from script_rag import KB, SchemeRetriever, optimise

# LangChain is optional at import time so the logic stays testable without it.
try:
    from langchain_core.tools import tool
    HAVE_LANGCHAIN = True
except ImportError:
    HAVE_LANGCHAIN = False

    class _Tool:
        def __init__(self, fn): self.fn, self.name, self.description = fn, fn.__name__, fn.__doc__
        def invoke(self, args): return self.fn(**args)

    def tool(fn): return _Tool(fn)

# ═════════════════════════════════════════════════════════════════════════════
# 1. SYNTHETIC 90-DAY UPI TELEMETRY
# ═════════════════════════════════════════════════════════════════════════════
ARCHETYPES = {
    "steady":    dict(p=.40, p_active=.86, mean=650, cv=.45, pool=8,  ess=.55, late=.06),
    "irregular": dict(p=.35, p_active=.50, mean=380, cv=1.00, pool=3,  ess=.78, late=.30),
    "mixed":     dict(p=.25, p_active=.70, mean=480, cv=.70, pool=12, ess=.66, late=.15),
}
OCCUPATIONS = ["delivery rider", "ride-app driver", "street vendor", "home-service worker"]
BILL_AMOUNT, LOW_BALANCE = 350, 100

def generate_upi_telemetry(n_workers=1500, days=90, seed=11):
    r = np.random.default_rng(seed)
    dates = pd.date_range(end=pd.Timestamp("2026-10-31"), periods=days)
    due_days = set(range(7, days, 15))
    names, probs = list(ARCHETYPES), [a["p"] for a in ARCHETYPES.values()]
    frames, meta = [], []
    for i in range(n_workers):
        wid, arch = f"W{i:04d}", r.choice(names, p=probs)
        c = ARCHETYPES[arch]
        p_act = float(np.clip(r.normal(c["p_active"], .06), .15, .98))
        mean = float(max(150, r.normal(c["mean"], c["mean"] * .12)))
        cv = float(max(.25, r.normal(c["cv"], .08)))
        pool = int(max(2, round(r.normal(c["pool"], 1.5))))
        ess_daily = float(np.clip(r.normal(c["ess"], .06), .25, 1)) * mean * p_act
        late = float(np.clip(r.normal(c["late"], .04), 0, .6))
        active = r.random(days) < p_act
        k = 1 / cv ** 2
        inflow = np.where(active, r.gamma(k, mean / k, days), 0.0)
        payer = np.where(active, r.integers(0, pool, days), -1)
        other = r.gamma(2, 25, days)
        bal, closing, ess_paid = 500.0, np.zeros(days), np.zeros(days)
        due, ontime = np.zeros(days, bool), np.zeros(days, bool)
        for d in range(days):
            avail = bal + inflow[d]
            if d in due_days:
                due[d] = True
                ontime[d] = bool(avail >= BILL_AMOUNT and r.random() > late)
                if ontime[d]: avail -= BILL_AMOUNT
            e = min(ess_daily, avail); avail -= e; ess_paid[d] = e
            avail -= min(other[d], avail)
            avail -= 0.7 * max(0.0, avail - 300)
            bal = avail; closing[d] = bal
        frames.append(pd.DataFrame({"worker_id": wid, "day": np.arange(days), "date": dates,
                                    "inflow": inflow, "payer_id": payer, "essential_out": ess_paid,
                                    "closing_balance": closing, "bill_due": due, "bill_on_time": ontime}))
        meta.append({"worker_id": wid, "archetype": arch,
                     "occupation": r.choice(OCCUPATIONS, p=[.35, .30, .20, .15]),
                     "has_bank_account": bool(r.random() < .85),
                     "consent": bool(r.random() < .97), "days_of_data": days})
    return pd.concat(frames, ignore_index=True), pd.DataFrame(meta).set_index("worker_id")

def engineer_features(tel: pd.DataFrame) -> pd.DataFrame:
    t = tel.assign(active=tel["inflow"] > 0, week=tel["day"] // 7)
    g = t.groupby("worker_id")
    weekly = t.groupby(["worker_id", "week"])["inflow"].sum().unstack()
    active_days = g["active"].sum().replace(0, np.nan)
    F = pd.DataFrame({
        "active_days_ratio": g["active"].mean(),
        "avg_daily_inflow": (g["inflow"].sum() / active_days).fillna(0),
        "inflow_cv": (weekly.std(axis=1) / weekly.mean(axis=1)).fillna(2),
        "payer_diversity": t[t["payer_id"] >= 0].groupby("worker_id")["payer_id"].nunique().reindex(g.size().index).fillna(0),
        "bill_on_time_ratio": (g["bill_on_time"].sum() / g["bill_due"].sum().replace(0, np.nan)).fillna(0),
        "low_balance_days_ratio": g["closing_balance"].apply(lambda s: (s < LOW_BALANCE).mean()),
        "essential_spend_share": (g["essential_out"].sum() / g["inflow"].sum().replace(0, np.nan)).clip(0, 1).fillna(1),
    })
    return F[FEATURES]

def synthetic_repayment_label(F: pd.DataFrame, seed=5) -> pd.Series:
    r = np.random.default_rng(seed)
    Z = (F - F.mean()) / F.std()
    w = dict(active_days_ratio=.5, avg_daily_inflow=.6, inflow_cv=-.7, payer_diversity=.1,
             bill_on_time_ratio=.6, low_balance_days_ratio=-.7, essential_spend_share=-.4)
    z = sum(w[c] * Z[c] for c in FEATURES) + r.normal(0, 1.6, len(F))
    return pd.Series((r.random(len(F)) < 1 / (1 + np.exp(-(.55 * z + .4)))).astype(int), index=F.index)

# ═════════════════════════════════════════════════════════════════════════════
# 2. THE THREE MODELS
# ═════════════════════════════════════════════════════════════════════════════
class SegmentationModel:
    NAMES = ["Steady daily earner", "Mixed-source earner", "Irregular earner"]

    def fit(self, F, k=3):
        self.sc = StandardScaler().fit(F[CLUSTER_FEATURES])
        Z = self.sc.transform(F[CLUSTER_FEATURES])
        self.km = KMeans(k, n_init=10, random_state=1).fit(Z)
        cols = {c: i for i, c in enumerate(CLUSTER_FEATURES)}
        stability = [Z[self.km.labels_ == c][:, cols["active_days_ratio"]].mean()
                     - Z[self.km.labels_ == c][:, cols["inflow_cv"]].mean() for c in range(k)]
        self.label = {c: self.NAMES[rank] for rank, c in enumerate(np.argsort(stability)[::-1])}
        return self

    def predict_many(self, F):
        Z = self.sc.transform(F[CLUSTER_FEATURES])
        return pd.Series([self.label[c] for c in self.km.predict(Z)], index=F.index)

    def predict(self, row):
        z = self.sc.transform(pd.DataFrame([row[CLUSTER_FEATURES]]))
        d = np.sort(self.km.transform(z)[0])
        c = int(self.km.predict(z)[0])
        return {"segment": self.label[c], "cluster_id": c,
                "confidence": round(float((d[1] - d[0]) / (d[1] + d[0] + 1e-9)), 2)}

class RepaymentModel:
    BANDS = [(0.45, "A"), (0.62, "B"), (0.78, "C"), (1.01, "D")]

    def fit(self, F, y, C=.5):
        Ftr, self.Fte, ytr, self.yte = train_test_split(F, y, test_size=.25, stratify=y, random_state=1)
        self.sc = StandardScaler().fit(Ftr[FEATURES])
        self.lr = LogisticRegression(C=C, max_iter=1000).fit(self.sc.transform(Ftr[FEATURES]), ytr)
        self.pte = self.lr.predict_proba(self.sc.transform(self.Fte[FEATURES]))[:, 1]
        self.metrics = {"AUC": round(roc_auc_score(self.yte, self.pte), 3),
                        "Brier": round(brier_score_loss(self.yte, self.pte), 3)}
        return self

    def band(self, p): return next(b for cut, b in self.BANDS if p < cut)

    def predict(self, row):
        x = self.sc.transform(pd.DataFrame([row[FEATURES]]))
        p = float(self.lr.predict_proba(x)[0, 1])
        contrib = self.lr.coef_[0] * x[0]
        order = np.argsort(contrib)
        helps = [FEATURES[i] for i in order[::-1][:2] if contrib[i] > 0]
        hurts = [FEATURES[i] for i in order[:2] if contrib[i] < 0]
        return {"p_repay": round(p, 3), "band": self.band(p),
                "helps": [REASON_TEXT[f][0] for f in helps], "hurts": [REASON_TEXT[f][1] for f in hurts]}

class MatchingEngine:
    def __init__(self, retriever): self.retriever = retriever

    def match(self, meta, feat, segment, score):
        credit_ok, kinds = POLICY[segment["segment"]][score["band"]]
        if not meta["has_bank_account"]:
            credit_ok = False
            kinds = list(dict.fromkeys(["enrolment"] + [k for k in kinds if k != "loan"]))
        monthly = feat["avg_daily_inflow"] * 30 * feat["active_days_ratio"]
        cap = MIN_REPAY_SHARE * monthly * (1 - min(feat["inflow_cv"], 1) * .5) if credit_ok else 0
        offers = self.retriever.search("", score["band"], meta["has_bank_account"],
                                       occupation=meta["occupation"], k=3, kinds=kinds)
        return {"segment": segment["segment"], "band": score["band"], "tier": score["band"],
                "credit_ok": bool(credit_ok), "lead_with": kinds, "max_monthly_repayment": int(round(cap)),
                "offers": [{"id": o["id"], "name": o["name"], "kind": o["kind"], "verified": o["verified"]} for o in offers],
                "reasons": score["helps"] + (score["hurts"] if score["band"] in ("A", "B") else []),
                "occupation": meta["occupation"], "has_bank_account": bool(meta["has_bank_account"])}

# ═════════════════════════════════════════════════════════════════════════════
# 3. CONTEXT + TOOLS
# ═════════════════════════════════════════════════════════════════════════════
@dataclass
class Context:
    features: pd.DataFrame
    meta: pd.DataFrame
    seg: SegmentationModel
    scorer: RepaymentModel
    matcher: MatchingEngine
    retriever: SchemeRetriever
    results: dict = field(default_factory=dict)

def build_context(n_workers=1500):
    tel, meta = generate_upi_telemetry(n_workers)
    F = engineer_features(tel)
    y = synthetic_repayment_label(F)
    retriever = SchemeRetriever(KB)
    return Context(F, meta, SegmentationModel().fit(F), RepaymentModel().fit(F, y),
                   MatchingEngine(retriever), retriever)

def build_tools(ctx: Context):
    @tool
    def segment_worker(worker_id: str) -> dict:
        """Assign a gig worker to a behavioural segment with K-Means on 90 days of UPI features.
        Input: worker_id. Returns segment name, cluster id and confidence."""
        out = ctx.seg.predict(ctx.features.loc[worker_id])
        ctx.results.setdefault(worker_id, {})["segment"] = out
        return out

    @tool
    def score_repayment(worker_id: str) -> dict:
        """Estimate repayment probability with ridge logistic regression on 90 days of UPI features.
        Input: worker_id. Returns probability, band A-D and plain-language reasons."""
        out = ctx.scorer.predict(ctx.features.loc[worker_id])
        ctx.results.setdefault(worker_id, {})["score"] = out
        return out

    @tool
    def match_offers(worker_id: str) -> dict:
        """Match the worker to allowed government schemes using rules, the segment x band policy
        matrix and scheme retrieval. Requires segment_worker and score_repayment to have run."""
        res = ctx.results.get(worker_id, {})
        if "segment" not in res or "score" not in res:
            return {"error": "segmentation and scoring must run first"}
        out = ctx.matcher.match(ctx.meta.loc[worker_id], ctx.features.loc[worker_id], res["segment"], res["score"])
        res["match"] = out
        return out

    @tool
    def write_script(worker_id: str) -> dict:
        """Write a plain-English script (target Flesch-Kincaid grade <= 6) from the matched offers,
        using only verified scheme facts. Requires match_offers to have run."""
        match = ctx.results.get(worker_id, {}).get("match")
        if not match:
            return {"error": "matching must run first"}
        r = optimise(match, ctx.retriever)
        last = [h for h in r.history if "grade" in h][-1]
        return {"script": r.script, "grade": last["grade"], "reading_ease": last["ease"],
                "grounded": r.grounded["ok"], "meaning_ok": r.meaning["ok"],
                "facts_used": [f["id" ] for f in r.facts], "safe_to_ship": r.safe_to_ship}

    return {"segmentation": segment_worker, "scoring": score_repayment,
            "matching": match_offers, "communication": write_script}

# ═════════════════════════════════════════════════════════════════════════════
# 4. AGENTS = LangChain tool-calling LLM (optional) around one tool each
# ═════════════════════════════════════════════════════════════════════════════
class PipelineState(TypedDict, total=False):
    worker_id: str
    blocked: str
    segmentation: dict
    scoring: dict
    matching: dict
    communication: dict
    audit: Annotated[list, operator.add]

SYSTEM = {
    "segmentation": "You are the Segmentation Agent for a gig-worker financial-inclusion service. Call segment_worker exactly once.",
    "scoring":      "You are the Scoring Agent. Call score_repayment exactly once.",
    "matching":     "You are the Matching Agent. Call match_offers exactly once.",
    "communication": "You are the Communication Agent. Call write_script exactly once.",
}
EXPLAIN = ("Now explain the tool result in ONE short sentence for a credit-risk reviewer. "
           "Use only values present in the tool result. Do not add numbers or promises.")

def _text(msg):
    c = getattr(msg, "content", msg)
    return c if isinstance(c, str) else " ".join(b.get("text", "") for b in c if isinstance(b, dict))

def _template(name, out):
    if "error" in out: return out["error"]
    return {
        "segmentation": lambda o: f"Behaviour pattern matches '{o.get('segment')}' (confidence {o.get('confidence')}).",
        "scoring":      lambda o: f"Repayment probability model estimates {o.get('p_repay') * 100:.1f}% chance (Band {o.get('band')}).",
        "matching":     lambda o: f"Eligible for {len(o.get('offers', []))} schemes (credit_ok: {o.get('credit_ok')}).",
        "communication": lambda o: f"Safe plain-English script generated (FK grade {o.get('grade')}, grounded: {o.get('grounded')})."
    }[name](out)

def make_agent(name, tool_obj, llm=None):
    if llm is None:
        def deterministic_node(state: PipelineState):
            wid = state["worker_id"]
            out = tool_obj.invoke({"worker_id": wid})
            return {name: out, "audit": [dict(agent=name, step="tool_call", output=out, reasoning="Deterministic mode - rules execution", timestamp=datetime.now(timezone.utc).isoformat())]}
        return deterministic_node

    bound = llm.bind_tools([tool_obj])

    def agent_node(state: PipelineState):
        wid = state["worker_id"]
        sys_msg = dict(role="system", content=SYSTEM[name])
        user_msg = dict(role="user", content=f"Process worker_id: {wid}")
        msgs = [sys_msg, user_msg]

        # Call bound LLM to invoke tool
        res = bound.invoke(msgs)
        calls = getattr(res, "tool_calls", [])
        if not calls:
            # fallback if model fails to call tool
            out = tool_obj.invoke({"worker_id": wid})
            expl = "Model failed tool call. Rule-based fallback executed."
        else:
            args = dict(calls[0]["args"])
            args["worker_id"] = wid # Force worker ID integrity
            out = tool_obj.invoke(args)

            # Explain phase
            msgs.append(res)
            msgs.append(dict(role="tool", content=json.dumps(out), tool_call_id=calls[0]["id"]))
            msgs.append(dict(role="user", content=EXPLAIN))
            expl_res = llm.invoke(msgs)
            expl = _text(expl_res)

        return {
            name: out,
            "audit": [dict(agent=name, step="complete", output=out, reasoning=expl, timestamp=datetime.now(timezone.utc).isoformat())]
        }
    return agent_node

# ═════════════════════════════════════════════════════════════════════════════
# 5. WORKFLOW COMPOSITION (LangGraph with strict gatekeeping)
# ═════════════════════════════════════════════════════════════════════════════
def gatekeeper(state: PipelineState, ctx: Context) -> dict:
    wid = state["worker_id"]
    meta = ctx.meta.loc[wid]
    if not meta["consent"]:
        return {"blocked": "Worker consent is missing."}
    if meta["days_of_data"] < 90:
        return {"blocked": f"Insufficient telemetry historical depth ({meta['days_of_data']} days)."}
    return {"blocked": ""}

def build_graph(ctx: Context, llm=None):
    try:
        from langgraph.graph import StateGraph, START, END
    except ImportError:
        print("[langgraph not installed -> orchestrating manually in code]"); return None

    tools = build_tools(ctx)
    builder = StateGraph(PipelineState)

    # Helper to encapsulate gatekeeper
    def gate_node(state: PipelineState):
        res = gatekeeper(state, ctx)
        return {**res, "audit": [dict(agent="gatekeeper", step="validate", output=res, reasoning=res["blocked"] or "Consent and data sufficiency validated", timestamp=datetime.now(timezone.utc).isoformat())]}

    builder.add_node("gatekeeper", gate_node)
    builder.add_node("segmentation", make_agent("segmentation", tools["segmentation"], llm))
    builder.add_node("scoring", make_agent("scoring", tools["scoring"], llm))
    builder.add_node("matching", make_agent("matching", tools["matching"], llm))
    builder.add_node("communication", make_agent("communication", tools["communication"], llm))

    builder.add_edge(START, "gatekeeper")

    def route_gate(state: PipelineState):
        return END if state.get("blocked") else "segmentation"

    builder.add_conditional_edges("gatekeeper", route_gate, {END: END, "segmentation": "segmentation"})
    builder.add_edge("segmentation", "scoring")
    builder.add_edge("scoring", "matching")
    builder.add_edge("matching", "communication")
    builder.add_edge("communication", END)

    return builder.compile()

# Model factory using modern supported gemini-3.8-flash model
def make_llm():
    if not os.environ.get("GEMINI_API_KEY"):
        print("[No GEMINI_API_KEY found -> deterministic mode]"); return None
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
    except ImportError:
        print("[langchain-google-genai not installed -> deterministic mode]"); return None
    return ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.0)

# ═════════════════════════════════════════════════════════════════════════════
# 6. GOVERNANCE AUDIT + RUNNER
# ═════════════════════════════════════════════════════════════════════════════
def run_pipeline(worker_id: str, ctx: Context, graph, llm=None):
    state = {"worker_id": worker_id, "audit": []}
    if graph is not None:
        res = graph.invoke(state)
        return res

    # Manual sequential flow fallback if LangGraph is missing
    res = gatekeeper(state, ctx)
    state.update(res)
    state["audit"].append(dict(agent="gatekeeper", step="validate", output=res, reasoning=res["blocked"] or "Consent and data sufficiency validated", timestamp=datetime.now(timezone.utc).isoformat()))
    if state.get("blocked"):
        return state

    tools = build_tools(ctx)
    for step in ["segmentation", "scoring", "matching", "communication"]:
        node = make_agent(step, tools[step], llm)
        out = node(state)
        state.update(out)
    return state

if __name__ == "__main__":
    ctx = build_context(100)
    llm = make_llm()
    graph = build_graph(ctx, llm)

    print(f"orchestrator: {'langgraph' if graph else 'manual'} | agents use LLM: {llm is not None}")
    print(f"models: Ridge-LR {ctx.scorer.metrics} | K-Means segments vs ground-truth archetypes:")

    F = ctx.features
    predicted_segs = ctx.seg.predict_many(F)
    print(pd.crosstab(predicted_segs, ctx.meta["archetype"]))

    # Process first worker as a demo
    demo_worker = ctx.meta.index[0]
    res = run_pipeline(demo_worker, ctx, graph, llm)
    print(f"\nProcessing worker {demo_worker} ({ctx.meta.loc[demo_worker]['occupation']}):")
    print(json.dumps(res, indent=2))

orchestrator: langgraph | agents use LLM: True
models: Ridge-LR {'AUC': np.float64(0.799), 'Brier': np.float64(0.177)} | K-Means segments vs ground-truth archetypes:
archetype            irregular  mixed  steady
row_0                                        
Irregular earner            27      0       0
Mixed-source earner          0     24       6
Steady daily earner          0      0      43

Processing worker W0000 (home-service worker):
{
  "worker_id": "W0000",
  "blocked": "",
  "segmentation": {
    "segment": "Steady daily earner",
    "cluster_id": 2,
    "confidence": 0.43
  },
  "scoring": {
    "p_repay": 0.943,
    "band": "D",
    "helps": [
      "your daily income is steady",
      "your balance rarely hits zero"
    ],
    "hurts": [
      "food, rent and bills use most of your money",
      "money comes from few sources"
    ]
  },
  "matching": {
    "segment": "Steady daily earner",
    "band": "D",
    "tier": "D",
    "credit_ok": true,
    "lead_with": [
      "lo